<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/16B_NeuroFHIR_SAFE_Runtime_Trace_and_FHIR_Conformance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 16B — NeuroFHIR-SAFE
## Executable Runtime Trace + FHIR Conformance Audit

This notebook is the **formal-to-running-system bridge**.

Notebook 16A answers: *what interaction sequences are possible in the specified finite model?*

Notebook 16B asks:
1. Does the deployed NeuroFHIR-Review implementation preserve the required stage order in its real event traces?
2. Do its observed transitions remain compatible with the verified Evidence-First architecture?
3. Do FHIR resources preserve review state, model identity, Task state, and provenance after persistence?

### Important distinction

P001/P002 dry-run traces are appropriate here as **engineering/runtime conformance traces**. They are **not human-study evidence** and must never be reported as participant outcomes.

In [1]:
# Cell 1 — Mount Drive / paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, re, hashlib, datetime, os
import pandas as pd

DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
WISH_ROOT = DRIVE_REPO_ROOT / "wish_extension"
SAFE_ROOT = WISH_ROOT / "neurofhir_safe"
FORMAL_RESULTS = SAFE_ROOT / "artifacts" / "formal" / "formal_results.json"
IMPLEMENTATION_ROOT = SAFE_ROOT / "artifacts" / "implementation"
IMPLEMENTATION_ROOT.mkdir(parents=True, exist_ok=True)

assert FORMAL_RESULTS.exists(), (
    "Run Notebook 16A first. Missing: " + str(FORMAL_RESULTS)
)
formal = json.loads(FORMAL_RESULTS.read_text(encoding="utf-8"))

print("Loaded formal core.")
print("Evidence-First config:", formal["evidence_first"]["config"])

Mounted at /content/drive
Loaded formal core.
Evidence-First config: EvidenceFirst.cfg


In [2]:
# Cell 2 — Discover QA/runtime JSON exports

# Search the WISH workspace for JSON files that actually contain event traces.
trace_candidates = []

for p in WISH_ROOT.rglob("*.json"):
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue

    if isinstance(obj, dict) and isinstance(obj.get("events"), list):
        trace_candidates.append((p, obj))

print("Trace exports discovered:", len(trace_candidates))

for p, obj in trace_candidates[:20]:
    print(
        " -",
        p,
        "| participant:",
        obj.get("participant_id"),
        "| events:",
        len(obj.get("events", []))
    )

assert trace_candidates, (
    "No JSON export containing an `events` list was found. "
    "Complete/export the P001/P002 dry run or place deterministic runtime "
    "trace exports under the WISH workspace, then rerun."
)

Trace exports discovered: 2
 - /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/evaluation/dry_run/SYNTHETIC_neurofhir_review_P001.json | participant: P001 | events: 144
 - /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/evaluation/dry_run/SYNTHETIC_neurofhir_review_P002.json | participant: P002 | events: 144


In [3]:
# Cell 3 — Normalize event records

def flat_text(x):
    if isinstance(x, dict):
        return " ".join(flat_text(v) for v in x.values())
    if isinstance(x, list):
        return " ".join(flat_text(v) for v in x)
    return str(x)

def event_case_id(e):
    for k in ["case_id","scenario_id","case","scenario"]:
        if isinstance(e, dict) and e.get(k) not in [None,""]:
            return str(e.get(k))
    return None

def event_condition(e):
    if not isinstance(e, dict):
        return None
    for k in ["condition","study_condition","sequence_condition"]:
        if e.get(k):
            return str(e[k])
    return None

def canonical_event(e):
    t = flat_text(e).lower()

    if ("initial" in t and any(x in t for x in ["submit","commit","judgment"])):
        return "COMMIT_INITIAL_JUDGMENT"

    if ("ai" in t and any(x in t for x in ["reveal","shown","show","visible","exposure","view"])):
        return "REVEAL_AI"

    if ("provenance" in t and any(x in t for x in ["open","view","inspect","shown"])):
        return "OPEN_PROVENANCE"

    if ("final" in t and any(x in t for x in ["submit","action","decision","disposition"])):
        return "FINAL_ACTION"

    if ("case" in t and any(x in t for x in ["start","open","begin"])):
        return "OPEN_CASE"

    return "OTHER"

normalized_rows = []

for path, obj in trace_candidates:
    participant = str(obj.get("participant_id") or "UNKNOWN")

    for i, e in enumerate(obj["events"]):
        normalized_rows.append({
            "source_file": str(path),
            "participant_id": participant,
            "event_index": i,
            "case_id": event_case_id(e),
            "condition": event_condition(e),
            "canonical_event": canonical_event(e),
            "raw_event": json.dumps(e, sort_keys=True),
        })

events_df = pd.DataFrame(normalized_rows)
display(events_df.head(30))

events_path = IMPLEMENTATION_ROOT / "normalized_runtime_events.csv"
events_df.to_csv(events_path, index=False)

print("✅ Normalized events:", events_path)

,source_file,participant_id,event_index,case_id,condition,canonical_event,raw_event
0,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,0,S02,ai-first,REVEAL_AI,"{""condition"": ""ai-first"", ""event_type"": ""case_..."
1,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,1,S02,ai-first,REVEAL_AI,"{""condition"": ""ai-first"", ""event_type"": ""scree..."
2,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,2,S02,ai-first,REVEAL_AI,"{""condition"": ""ai-first"", ""event_type"": ""evide..."
3,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,3,S02,ai-first,REVEAL_AI,"{""ai_visible_before_initial_judgment"": true, ""..."
4,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,4,S02,ai-first,COMMIT_INITIAL_JUDGMENT,"{""condition"": ""ai-first"", ""event_type"": ""scree..."
5,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,5,S02,ai-first,COMMIT_INITIAL_JUDGMENT,"{""ai_visible_before_initial_judgment"": true, ""..."
6,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,6,S02,ai-first,REVEAL_AI,"{""condition"": ""ai-first"", ""event_type"": ""scree..."
7,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,7,S02,ai-first,REVEAL_AI,"{""condition"": ""ai-first"", ""event_type"": ""scree..."
8,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,8,S02,ai-first,REVEAL_AI,"{""condition"": ""ai-first"", ""event_type"": ""passp..."
9,/content/drive/MyDrive/neurofhir-qc/wish_exten...,P001,9,S02,ai-first,REVEAL_AI,"{""condition"": ""ai-first"", ""event_type"": ""prove..."


✅ Normalized events: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/artifacts/implementation/normalized_runtime_events.csv


In [4]:
# Cell 4 — Evidence-First / AI-First ordering audit

# We audit each participant × case with recognizable study-stage events.
# If case_id is absent in an event, the trace cannot support case-level ordering
# and is reported as incomplete rather than guessed.

audits = []

usable = events_df.dropna(subset=["case_id"]).copy()

for (participant, case_id), g in usable.groupby(["participant_id","case_id"], dropna=False):
    g = g.sort_values("event_index")

    positions = {}
    for event_name in [
        "COMMIT_INITIAL_JUDGMENT",
        "REVEAL_AI",
        "OPEN_PROVENANCE",
        "FINAL_ACTION",
    ]:
        xs = g.loc[g["canonical_event"] == event_name, "event_index"].tolist()
        positions[event_name] = xs[0] if xs else None

    # Prefer condition attached to events if available.
    conditions = [x for x in g["condition"].dropna().astype(str).tolist() if x]
    condition = conditions[0] if conditions else None

    initial_i = positions["COMMIT_INITIAL_JUDGMENT"]
    ai_i = positions["REVEAL_AI"]
    final_i = positions["FINAL_ACTION"]

    complete_for_order = initial_i is not None and ai_i is not None

    evidence_first_order = (
        complete_for_order and initial_i < ai_i
    )
    ai_first_order = (
        complete_for_order and ai_i < initial_i
    )

    # If an explicit condition is unavailable, record the observed architecture
    # without pretending to know the randomized label.
    observed_architecture = (
        "Evidence-First" if evidence_first_order
        else "AI-First" if ai_first_order
        else "Incomplete/ambiguous"
    )

    final_after_ai = (
        final_i is not None and ai_i is not None and ai_i < final_i
    )

    audits.append({
        "participant_id": participant,
        "case_id": case_id,
        "declared_condition": condition,
        "observed_architecture": observed_architecture,
        "initial_event_index": initial_i,
        "ai_event_index": ai_i,
        "provenance_event_index": positions["OPEN_PROVENANCE"],
        "final_event_index": final_i,
        "order_trace_complete": complete_for_order,
        "human_first_order_observed": evidence_first_order,
        "ai_first_order_observed": ai_first_order,
        "final_after_ai": final_after_ai,
    })

order_df = pd.DataFrame(audits)
display(order_df)

order_path = IMPLEMENTATION_ROOT / "runtime_trace_audit.csv"
order_df.to_csv(order_path, index=False)

print("Saved:", order_path)

,participant_id,case_id,declared_condition,observed_architecture,initial_event_index,ai_event_index,provenance_event_index,final_event_index,order_trace_complete,human_first_order_observed,ai_first_order_observed,final_after_ai
0,P001,S01,evidence-first,AI-First,135,132,None,None,True,False,True,False
1,P001,S02,ai-first,AI-First,4,0,None,None,True,False,True,False
2,P001,S03,ai-first,AI-First,16,12,None,None,True,False,True,False
3,P001,S04,evidence-first,AI-First,75,72,None,None,True,False,True,False
4,P001,S05,evidence-first,AI-First,63,60,None,None,True,False,True,False
5,P001,S06,ai-first,AI-First,52,48,None,None,True,False,True,False
6,P001,S07,ai-first,AI-First,40,36,None,None,True,False,True,False
7,P001,S08,evidence-first,AI-First,27,24,None,None,True,False,True,False
8,P001,S09,evidence-first,AI-First,111,108,None,None,True,False,True,False
9,P001,S10,ai-first,AI-First,88,84,None,None,True,False,True,False


Saved: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/artifacts/implementation/runtime_trace_audit.csv


In [5]:
# Cell 5 — Runtime conformance metrics

complete_order = order_df[order_df["order_trace_complete"]].copy()

assert len(complete_order) > 0, (
    "No case trace contains both initial-judgment and AI-reveal events. "
    "This runtime evidence is insufficient for ordering conformance."
)

# Primary implementation endpoint for the Evidence-First traces:
# every trace empirically classified as Evidence-First must satisfy initial < AI.
ef = complete_order[
    complete_order["observed_architecture"] == "Evidence-First"
].copy()

af = complete_order[
    complete_order["observed_architecture"] == "AI-First"
].copy()

runtime_summary = {
    "trace_cases_with_order_evidence": int(len(complete_order)),
    "evidence_first_traces": int(len(ef)),
    "ai_first_traces": int(len(af)),
    "evidence_first_order_conformance_rate": (
        float(ef["human_first_order_observed"].mean()) if len(ef) else None
    ),
    "final_after_ai_rate": (
        float(
            order_df.dropna(subset=["final_event_index"])["final_after_ai"].mean()
        )
        if order_df["final_event_index"].notna().any() else None
    ),
}

print(json.dumps(runtime_summary, indent=2))

{
  "trace_cases_with_order_evidence": 24,
  "evidence_first_traces": 0,
  "ai_first_traces": 24,
  "evidence_first_order_conformance_rate": null,
  "final_after_ai_rate": null
}


In [6]:
# Cell 6 — Discover FHIR JSON artifacts

def collect_resources(obj):
    out = []

    if isinstance(obj, dict):
        if isinstance(obj.get("resourceType"), str):
            out.append(obj)

        # Bundle entries
        if obj.get("resourceType") == "Bundle":
            for entry in obj.get("entry", []):
                if isinstance(entry, dict) and isinstance(entry.get("resource"), dict):
                    out.extend(collect_resources(entry["resource"]))

        # Also scan nested objects in case resources are wrapped.
        for k, v in obj.items():
            if k == "entry" and obj.get("resourceType") == "Bundle":
                continue
            if isinstance(v, (dict, list)):
                out.extend(collect_resources(v))

    elif isinstance(obj, list):
        for x in obj:
            out.extend(collect_resources(x))

    return out

fhir_files = []

for p in DRIVE_REPO_ROOT.rglob("*.json"):
    try:
        obj = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue

    resources = collect_resources(obj)
    resource_types = sorted(set(r.get("resourceType") for r in resources if r.get("resourceType")))

    if any(
        rt in resource_types
        for rt in ["Observation","DiagnosticReport","Task","Device","Provenance","ImagingStudy"]
    ):
        fhir_files.append((p, resources, resource_types))

print("FHIR-bearing JSON files discovered:", len(fhir_files))
for p, _, types in fhir_files[:30]:
    print(" -", p, "|", ", ".join(types))

FHIR-bearing JSON files discovered: 152
 - /content/drive/MyDrive/neurofhir-qc/wish_extension/data/neurofhir_qc_app_data_snapshot.json | DiagnosticReport, Observation, Practitioner, Provenance, Task
 - /content/drive/MyDrive/neurofhir-qc/app/frontend/dist/data/app_data.json | DiagnosticReport, Observation, Practitioner, Provenance, Task
 - /content/drive/MyDrive/neurofhir-qc/app/frontend/public/data/app_data.json | DiagnosticReport, Observation, Practitioner, Provenance, Task
 - /content/drive/MyDrive/neurofhir-qc/app/backend/data/app_data.json | DiagnosticReport, Observation, Practitioner, Provenance, Task
 - /content/drive/MyDrive/neurofhir-qc/submission/final_release/AMIA_FHIR_Resources/master_evidence_collection_bundle.json | Bundle, Device, DiagnosticReport, Observation, Provenance, Task
 - /content/drive/MyDrive/neurofhir-qc/submission/final_release/AMIA_FHIR_Resources/master_review_evidence_bundle.json | Bundle, DiagnosticReport, Observation, Practitioner, Provenance, Task
 - /c

In [7]:
# Cell 7 — FHIR safety audit

def rid(r):
    return f"{r.get('resourceType','?')}/{r.get('id','?')}"

fhir_rows = []

for path, resources, resource_types in fhir_files:
    by_type = {}
    for r in resources:
        by_type.setdefault(r.get("resourceType","UNKNOWN"), []).append(r)

    observations = by_type.get("Observation", [])
    reports = by_type.get("DiagnosticReport", [])
    tasks = by_type.get("Task", [])
    devices = by_type.get("Device", [])
    provenances = by_type.get("Provenance", [])

    terminal_results = [
        r for r in observations + reports
        if str(r.get("status","")).lower() in {"final","entered-in-error"}
    ]
    final_results = [
        r for r in observations + reports
        if str(r.get("status","")).lower() == "final"
    ]
    rejected_results = [
        r for r in observations + reports
        if str(r.get("status","")).lower() == "entered-in-error"
    ]

    task_statuses = [str(t.get("status","")).lower() for t in tasks]

    # Provenance targets may be direct references or wrapped references.
    prov_text = json.dumps(provenances, sort_keys=True).lower()
    terminal_targeted = True
    missing_prov_targets = []

    for r in terminal_results:
        if r.get("id"):
            ref = rid(r).lower()
            if ref not in prov_text and str(r.get("id")).lower() not in prov_text:
                terminal_targeted = False
                missing_prov_targets.append(rid(r))

    checks = {
        "has_device_identity": (len(devices) > 0) if terminal_results else True,
        "terminal_results_have_provenance": terminal_targeted if terminal_results else True,
        "final_has_completed_task": (
            ("completed" in task_statuses) if final_results else True
        ),
        "rejected_has_rejected_task": (
            ("rejected" in task_statuses) if rejected_results else True
        ),
    }

    for check_name, passed in checks.items():
        fhir_rows.append({
            "source_file": str(path),
            "check": check_name,
            "passed": bool(passed),
            "resource_types": ",".join(resource_types),
            "terminal_results": len(terminal_results),
            "details": (
                "missing provenance target(s): " + ",".join(missing_prov_targets)
                if check_name == "terminal_results_have_provenance" and missing_prov_targets
                else ""
            ),
        })

fhir_df = pd.DataFrame(fhir_rows)

if len(fhir_df):
    display(fhir_df)
    fhir_rate = float(fhir_df["passed"].mean())
else:
    fhir_rate = None
    print(
        "⚠️ No FHIR-bearing JSON artifacts were found. "
        "FHIR conformance remains INCOMPLETE until archived/persisted resources "
        "are placed in the repo/Drive workspace."
    )

fhir_path = IMPLEMENTATION_ROOT / "fhir_safety_audit.csv"
fhir_df.to_csv(fhir_path, index=False)

print("FHIR audit:", fhir_path)

,source_file,check,passed,resource_types,terminal_results,details
0,/content/drive/MyDrive/neurofhir-qc/wish_exten...,has_device_identity,False,"DiagnosticReport,Observation,Practitioner,Prov...",6,
1,/content/drive/MyDrive/neurofhir-qc/wish_exten...,terminal_results_have_provenance,True,"DiagnosticReport,Observation,Practitioner,Prov...",6,
2,/content/drive/MyDrive/neurofhir-qc/wish_exten...,final_has_completed_task,True,"DiagnosticReport,Observation,Practitioner,Prov...",6,
3,/content/drive/MyDrive/neurofhir-qc/wish_exten...,rejected_has_rejected_task,True,"DiagnosticReport,Observation,Practitioner,Prov...",6,
4,/content/drive/MyDrive/neurofhir-qc/app/fronte...,has_device_identity,False,"DiagnosticReport,Observation,Practitioner,Prov...",6,
...,...,...,...,...,...,...
603,/content/drive/MyDrive/neurofhir-qc/data/synth...,rejected_has_rejected_task,True,Observation,1,
604,/content/drive/MyDrive/neurofhir-qc/data/synth...,has_device_identity,False,Observation,1,
605,/content/drive/MyDrive/neurofhir-qc/data/synth...,terminal_results_have_provenance,False,Observation,1,missing provenance target(s): Observation/prio...
606,/content/drive/MyDrive/neurofhir-qc/data/synth...,final_has_completed_task,False,Observation,1,


FHIR audit: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/artifacts/implementation/fhir_safety_audit.csv


In [8]:
# Cell 8 — Implementation/FHIR summary

implementation_summary = {
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "formal_core_loaded": True,
    "runtime": runtime_summary,
    "fhir": {
        "files_audited": int(len(fhir_files)),
        "assertions": int(len(fhir_df)),
        "pass_rate": fhir_rate,
    },
    "claim_boundary": (
        "Runtime traces are system-conformance evidence only. "
        "P001/P002 or scripted traces are not human behavioral data."
    ),
}

summary_path = IMPLEMENTATION_ROOT / "implementation_summary.json"
summary_path.write_text(
    json.dumps(implementation_summary, indent=2),
    encoding="utf-8"
)

print("✅ Summary:", summary_path)

✅ Summary: /content/drive/MyDrive/neurofhir-qc/wish_extension/neurofhir_safe/artifacts/implementation/implementation_summary.json


In [9]:
# Cell 9 — Core conformance gate

runtime_ok = (
    runtime_summary["evidence_first_traces"] > 0
    and runtime_summary["evidence_first_order_conformance_rate"] == 1.0
)

# Strongest core requires FHIR artifacts; if not present, this gate remains false
# rather than silently calling the study complete.
fhir_ok = (
    fhir_rate is not None
    and len(fhir_df) > 0
    and bool(fhir_df["passed"].all())
)

print("Runtime Evidence-First order conformance:", "PASS" if runtime_ok else "INCOMPLETE/FAIL")
print("FHIR audit completeness:", "PASS" if fhir_ok else "INCOMPLETE/FAIL")

if runtime_ok and fhir_ok:
    print("=" * 82)
    print("✅ NOTEBOOK 16B IMPLEMENTATION/FHIR CONFORMANCE GATE: TRUE")
    print("=" * 82)
else:
    print("=" * 82)
    print("⚠️ NOTEBOOK 16B CORE GATE IS NOT YET COMPLETE")
    print("=" * 82)
    print("Do not report the strongest NeuroFHIR-SAFE end-to-end claim until both")
    print("runtime trace conformance and FHIR persistence/audit checks pass.")

Runtime Evidence-First order conformance: INCOMPLETE/FAIL
FHIR audit completeness: INCOMPLETE/FAIL
⚠️ NOTEBOOK 16B CORE GATE IS NOT YET COMPLETE
Do not report the strongest NeuroFHIR-SAFE end-to-end claim until both
runtime trace conformance and FHIR persistence/audit checks pass.
